In [4]:
# ============================================================
# Pull GitHub Actions telemetry timeline + raw job log
#
# Target:
# https://github.com/FooIbar/EhViewer/actions/runs/22557070276/job/65336353665
#
# Outputs:
#   github_run_general_timeline_65336353665.csv
#   github_job_log_65336353665.txt
#
# Token:
#   Loaded from:
#   C:\GitHub\Android-Mobile-Apps\All_Tokens.env
#
#   Expected variable:
#   GITHUB_TOKEN_1=ghp_...
# ============================================================

from pathlib import Path
from urllib.parse import urlparse
import os
import time
import requests
import pandas as pd


# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------
RUN_URL = "https://github.com/FooIbar/EhViewer/actions/runs/22557070276/job/65336353665"

ENV_FILE = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

OUT_DIR = Path(
    r"C:\Android Mobile App\ICST2026_Ext\0.2-Validation"
    r"\Step-2-Stratified_Sample\Manual_Checks\Manual_Review_Run_65336353665"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIMELINE_OUT_FILE = OUT_DIR / "github_run_general_timeline_65336353665.csv"


# ------------------------------------------------------------
# Load token
# ------------------------------------------------------------
def load_env_file(env_file: Path):
    """
    Load key=value pairs from a .env-style file without requiring python-dotenv.
    Supports lines like:
      GITHUB_TOKEN_1=ghp_xxx
    """
    if not env_file.exists():
        raise FileNotFoundError(f"Env file not found: {env_file}")

    with env_file.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            if "=" not in line:
                continue

            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")

            os.environ[key] = value


def get_github_token():
    """
    Prefer GITHUB_TOKEN_1 from All_Tokens.env.
    Fall back to GITHUB_TOKEN if needed.
    """
    load_env_file(ENV_FILE)

    token = os.getenv("GITHUB_TOKEN_1") or os.getenv("GITHUB_TOKEN")

    if not token:
        raise RuntimeError(
            "No GitHub token found. Expected GITHUB_TOKEN_1 or GITHUB_TOKEN "
            f"in {ENV_FILE}"
        )

    print("GitHub token loaded:", token[:6] + "..." + token[-4:])
    return token


GITHUB_TOKEN = get_github_token()


# ------------------------------------------------------------
# URL parsers
# ------------------------------------------------------------
def parse_github_run_url(run_url: str):
    """
    Extract owner, repo, and run_id from either:
      https://github.com/{owner}/{repo}/actions/runs/{run_id}
    or:
      https://github.com/{owner}/{repo}/actions/runs/{run_id}/job/{job_id}
    """
    parsed = urlparse(run_url)
    parts = parsed.path.strip("/").split("/")

    if len(parts) < 5 or parts[2:4] != ["actions", "runs"]:
        raise ValueError(f"Invalid GitHub Actions run URL: {run_url}")

    owner = parts[0]
    repo = parts[1]
    run_id = parts[4]

    return owner, repo, run_id


def parse_github_job_url(run_url: str):
    """
    Extract owner, repo, run_id, and job_id from:
      https://github.com/{owner}/{repo}/actions/runs/{run_id}/job/{job_id}
    """
    parsed = urlparse(run_url)
    parts = parsed.path.strip("/").split("/")

    if len(parts) < 7 or parts[2:4] != ["actions", "runs"] or parts[5] != "job":
        raise ValueError(
            "This URL does not include a job id. Expected format:\n"
            "https://github.com/{owner}/{repo}/actions/runs/{run_id}/job/{job_id}"
        )

    owner = parts[0]
    repo = parts[1]
    run_id = parts[4]
    job_id = parts[6]

    return owner, repo, run_id, job_id


# ------------------------------------------------------------
# GitHub API helpers
# ------------------------------------------------------------
def github_headers(user_agent: str = "github-actions-run-telemetry-review"):
    return {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": user_agent,
    }


def github_get_json(url: str, params=None):
    headers = github_headers()

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=60,
    )

    # Basic rate-limit handling
    if response.status_code == 403 and response.headers.get("X-RateLimit-Remaining") == "0":
        reset_epoch = int(response.headers.get("X-RateLimit-Reset", "0"))
        sleep_seconds = max(0, reset_epoch - int(time.time()) + 2)
        print(f"Rate limited. Sleeping for {sleep_seconds} seconds...")
        time.sleep(sleep_seconds)

        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=60,
        )

    if response.status_code >= 400:
        print("\nGitHub API request failed")
        print("URL:", url)
        print("Status:", response.status_code)
        print("Response:", response.text[:1000])

    response.raise_for_status()
    return response.json()


def github_paginated_get(url: str, item_key: str):
    all_items = []
    page = 1

    while True:
        payload = github_get_json(
            url,
            params={
                "per_page": 100,
                "page": page,
            },
        )

        batch = payload.get(item_key, [])
        all_items.extend(batch)

        if len(batch) < 100:
            break

        page += 1

    return all_items


# ------------------------------------------------------------
# Time helpers
# ------------------------------------------------------------
def dt(x):
    return pd.to_datetime(x, errors="coerce", utc=True)


def seconds_between(start, end):
    start = dt(start)
    end = dt(end)

    if pd.isna(start) or pd.isna(end):
        return None

    return (end - start).total_seconds()


def seconds_from_run_start(timestamp, run_start):
    timestamp = dt(timestamp)
    run_start = dt(run_start)

    if pd.isna(timestamp) or pd.isna(run_start):
        return None

    return (timestamp - run_start).total_seconds()


# ------------------------------------------------------------
# Pull structured run/job/step telemetry
# ------------------------------------------------------------
owner, repo, run_id = parse_github_run_url(RUN_URL)
owner_for_log, repo_for_log, run_id_for_log, job_id_for_log = parse_github_job_url(RUN_URL)

api_base = "https://api.github.com"
run_api_url = f"{api_base}/repos/{owner}/{repo}/actions/runs/{run_id}"
jobs_api_url = f"{api_base}/repos/{owner}/{repo}/actions/runs/{run_id}/jobs"

run = github_get_json(run_api_url)
jobs = github_paginated_get(jobs_api_url, item_key="jobs")

run_start = run.get("run_started_at") or run.get("created_at")
run_end = run.get("updated_at")

print("\n=== Run Metadata ===")
print(f"Repository: {owner}/{repo}")
print(f"Run ID: {run_id}")
print(f"Job ID for log download: {job_id_for_log}")
print(f"Workflow: {run.get('name')}")
print(f"Run status/conclusion: {run.get('status')} / {run.get('conclusion')}")
print(f"Run start: {run_start}")
print(f"Run end:   {run_end}")
print(f"Jobs pulled: {len(jobs)}")


# ------------------------------------------------------------
# Build timeline table
# ------------------------------------------------------------
timeline_rows = []

# Run start row
timeline_rows.append({
    "event_type": "run",
    "event_name": "RUN START",
    "repo": f"{owner}/{repo}",
    "run_id": run.get("id"),
    "workflow_name": run.get("name"),
    "workflow_path": run.get("path"),
    "run_attempt": run.get("run_attempt"),
    "run_event": run.get("event"),
    "branch": run.get("head_branch"),
    "sha": run.get("head_sha"),
    "job_id": None,
    "job_name": None,
    "step_number": None,
    "step_name": None,
    "status": run.get("status"),
    "conclusion": run.get("conclusion"),
    "started_at": run_start,
    "completed_at": run_end,
    "duration_seconds": seconds_between(run_start, run_end),
    "seconds_from_run_start": 0,
    "html_url": run.get("html_url"),
})

# Job and step rows
for job in jobs:
    job_id = job.get("id")
    job_name = job.get("name")
    job_start = job.get("started_at")
    job_end = job.get("completed_at")

    timeline_rows.append({
        "event_type": "job",
        "event_name": f"JOB: {job_name}",
        "repo": f"{owner}/{repo}",
        "run_id": run.get("id"),
        "workflow_name": run.get("name"),
        "workflow_path": run.get("path"),
        "run_attempt": run.get("run_attempt"),
        "run_event": run.get("event"),
        "branch": run.get("head_branch"),
        "sha": run.get("head_sha"),
        "job_id": job_id,
        "job_name": job_name,
        "step_number": None,
        "step_name": None,
        "status": job.get("status"),
        "conclusion": job.get("conclusion"),
        "started_at": job_start,
        "completed_at": job_end,
        "duration_seconds": seconds_between(job_start, job_end),
        "seconds_from_run_start": seconds_from_run_start(job_start, run_start),
        "html_url": job.get("html_url"),
    })

    for step in job.get("steps", []):
        step_name = step.get("name")
        step_start = step.get("started_at")
        step_end = step.get("completed_at")

        timeline_rows.append({
            "event_type": "step",
            "event_name": f"STEP: {job_name} / {step_name}",
            "repo": f"{owner}/{repo}",
            "run_id": run.get("id"),
            "workflow_name": run.get("name"),
            "workflow_path": run.get("path"),
            "run_attempt": run.get("run_attempt"),
            "run_event": run.get("event"),
            "branch": run.get("head_branch"),
            "sha": run.get("head_sha"),
            "job_id": job_id,
            "job_name": job_name,
            "step_number": step.get("number"),
            "step_name": step_name,
            "status": step.get("status"),
            "conclusion": step.get("conclusion"),
            "started_at": step_start,
            "completed_at": step_end,
            "duration_seconds": seconds_between(step_start, step_end),
            "seconds_from_run_start": seconds_from_run_start(step_start, run_start),
            "html_url": job.get("html_url"),
        })

# Run end row
timeline_rows.append({
    "event_type": "run",
    "event_name": "RUN END",
    "repo": f"{owner}/{repo}",
    "run_id": run.get("id"),
    "workflow_name": run.get("name"),
    "workflow_path": run.get("path"),
    "run_attempt": run.get("run_attempt"),
    "run_event": run.get("event"),
    "branch": run.get("head_branch"),
    "sha": run.get("head_sha"),
    "job_id": None,
    "job_name": None,
    "step_number": None,
    "step_name": None,
    "status": run.get("status"),
    "conclusion": run.get("conclusion"),
    "started_at": run_end,
    "completed_at": run_end,
    "duration_seconds": 0,
    "seconds_from_run_start": seconds_from_run_start(run_end, run_start),
    "html_url": run.get("html_url"),
})

timeline_df = pd.DataFrame(timeline_rows)

timeline_df["_sort_started_at"] = pd.to_datetime(
    timeline_df["started_at"],
    errors="coerce",
    utc=True,
)

timeline_df["_event_order"] = timeline_df["event_type"].map({
    "run": 0,
    "job": 1,
    "step": 2,
}).fillna(9)

timeline_df = (
    timeline_df
    .sort_values(
        by=["_sort_started_at", "_event_order", "job_name", "step_number"],
        na_position="last",
    )
    .drop(columns=["_sort_started_at", "_event_order"])
)

timeline_df.to_csv(TIMELINE_OUT_FILE, index=False)

print("\nSaved general timeline to:")
print(TIMELINE_OUT_FILE)


# ------------------------------------------------------------
# Download raw GitHub Actions job log
# ------------------------------------------------------------
def github_download_job_log(owner: str, repo: str, job_id: str, out_dir: Path):
    """
    Download raw log text for a GitHub Actions job.

    Endpoint:
      GET /repos/{owner}/{repo}/actions/jobs/{job_id}/logs

    GitHub normally returns a redirect to a temporary signed URL.
    requests follows it automatically.
    """
    log_api_url = f"https://api.github.com/repos/{owner}/{repo}/actions/jobs/{job_id}/logs"

    out_file = out_dir / f"github_job_log_{job_id}.txt"
    error_file = out_dir / f"github_job_log_{job_id}_ERROR.txt"

    headers = github_headers(user_agent="github-actions-job-log-review")

    response = requests.get(
        log_api_url,
        headers=headers,
        timeout=120,
        allow_redirects=True,
    )

    if response.status_code == 200:
        out_file.write_bytes(response.content)
        print("\nSaved raw job log to:")
        print(out_file)
        return out_file

    msg = (
        f"Raw job log could not be downloaded.\n"
        f"Repository: {owner}/{repo}\n"
        f"Job ID: {job_id}\n"
        f"API URL: {log_api_url}\n"
        f"HTTP status: {response.status_code}\n\n"
        f"Response headers:\n{dict(response.headers)}\n\n"
        f"Response text:\n{response.text[:4000]}\n\n"
        f"Notes:\n"
        f"- If HTTP status is 403, the token is probably missing Actions: read access, "
        f"or the token is not authorized for this repository.\n"
        f"- If HTTP status is 404 or 410, the raw log may be expired or unavailable.\n"
        f"- Structured job/step telemetry may still be available even when raw logs are blocked.\n"
    )

    error_file.write_text(msg, encoding="utf-8")

    print("\nCould not download raw job log. Saved explanation to:")
    print(error_file)

    # Do not crash the whole script; timeline extraction is still useful.
    return None


github_download_job_log(
    owner=owner_for_log,
    repo=repo_for_log,
    job_id=job_id_for_log,
    out_dir=OUT_DIR,
)


# ------------------------------------------------------------
# Console view
# ------------------------------------------------------------
print("\n=== General Run Timeline ===")

display_cols = [
    "event_type",
    "job_id",
    "job_name",
    "step_number",
    "step_name",
    "status",
    "conclusion",
    "started_at",
    "completed_at",
    "duration_seconds",
    "seconds_from_run_start",
]

print(
    timeline_df[display_cols]
    .to_string(index=False, max_colwidth=80)
)

GitHub token loaded: ghp_mH...D1rN

=== Run Metadata ===
Repository: FooIbar/EhViewer
Run ID: 22557070276
Job ID for log download: 65336353665
Workflow: Baseline profile generation
Run status/conclusion: completed / success
Run start: 2026-03-02T00:55:01Z
Run end:   2026-03-02T01:14:54Z
Jobs pulled: 1

Saved general timeline to:
C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-2-Stratified_Sample\Manual_Checks\Manual_Review_Run_65336353665\github_run_general_timeline_65336353665.csv

Saved raw job log to:
C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-2-Stratified_Sample\Manual_Checks\Manual_Review_Run_65336353665\github_job_log_65336353665.txt

=== General Run Timeline ===
event_type       job_id         job_name  step_number                 step_name    status conclusion           started_at         completed_at  duration_seconds  seconds_from_run_start
       run          NaN             None          NaN                      None completed    success 2026-03-02T00:55: